# Backward Model Hyperparameter Tuning (Wrought only)

Tunes GMM (n_components, covariance_type) using BIC on the **wrought** composition space. Saves best params to `backward.wrought.GMM` in `hyperparams_config.json`.

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.mixture import GaussianMixture, BayesianGaussianMixture
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings('ignore')

try:
    from utils import load_hyperparams, save_hyperparams, get_default_hyperparams
except ImportError:
    import sys
    sys.path.insert(0, os.getcwd())
    from utils import load_hyperparams, save_hyperparams, get_default_hyperparams

INPUT_COLS = ['Al', 'Si', 'Fe', 'Cu', 'Mn', 'Mg', 'Cr', 'Ni', 'Zn', 'Ga', 'V', 'Ti']

In [2]:
# Load wrought composition data (12 element columns only)
WROUGHT_PATH = 'wrought_alloys_final.csv'

def load_wrought_compositions(path):
    if not os.path.exists(path):
        return None
    if path.endswith('.xlsx') or path.endswith('.xls'):
        df = pd.read_excel(path)
    else:
        try:
            with open(path, 'rb') as f:
                if f.read(2) == b'PK':
                    df = pd.read_excel(path)
                else:
                    try:
                        df = pd.read_csv(path, encoding='utf-8')
                    except UnicodeDecodeError:
                        df = pd.read_csv(path, encoding='latin-1')
        except Exception:
            df = pd.read_excel(path)
    for c in INPUT_COLS:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    df[INPUT_COLS] = df[INPUT_COLS].fillna(0.0)
    return df[INPUT_COLS].dropna()

X_wrought = load_wrought_compositions(WROUGHT_PATH)
if X_wrought is not None:
    print(f'Wrought composition data: {X_wrought.shape}')
else:
    print('Wrought data not found.')

Wrought composition data: (868, 12)


In [3]:
# Tune GMM and BGMM for wrought composition generation
n_components_range = [6, 8, 10, 12, 15, 20]
covariance_types = ['full', 'tied']
bgmm_prior_range = [0.05, 0.1, 0.2]


def tune_gmm_bic(X, dataset_name):
    if X is None or len(X) < 20:
        return None
    best_bic = np.inf
    best_params = None
    results = []
    for n in n_components_range:
        for cov in covariance_types:
            try:
                gmm = GaussianMixture(n_components=n, covariance_type=cov, random_state=42)
                gmm.fit(X)
                bic = gmm.bic(X)
                results.append({'n_components': n, 'covariance_type': cov, 'bic': bic})
                if bic < best_bic:
                    best_bic = bic
                    best_params = {'n_components': n, 'covariance_type': cov, 'random_state': 42}
            except Exception as e:
                print(f'  Skip n={n} cov={cov}: {e}')
    print(f'{dataset_name} GMM BIC (top 3):')
    for r in sorted(results, key=lambda x: x['bic'])[:3]:
        print(f"  n={r['n_components']}, cov={r['covariance_type']}: BIC={r['bic']:.0f}")
    print(f'  Best: {best_params}\n')
    return best_params


def tune_bgmm_holdout(X, dataset_name):
    if X is None or len(X) < 40:
        return None

    X_train, X_val = train_test_split(X, test_size=0.2, random_state=42)
    best_score = -np.inf
    best_params = None
    results = []

    for n in n_components_range:
        for cov in covariance_types:
            for prior in bgmm_prior_range:
                try:
                    bgmm = BayesianGaussianMixture(
                        n_components=n,
                        covariance_type=cov,
                        weight_concentration_prior_type='dirichlet_process',
                        weight_concentration_prior=prior,
                        max_iter=1000,
                        random_state=42,
                    )
                    bgmm.fit(X_train)
                    val_score = float(bgmm.score(X_val))
                    results.append({
                        'n_components': n,
                        'covariance_type': cov,
                        'weight_concentration_prior': prior,
                        'val_avg_log_likelihood': val_score,
                    })
                    if val_score > best_score:
                        best_score = val_score
                        best_params = {
                            'n_components': n,
                            'covariance_type': cov,
                            'weight_concentration_prior_type': 'dirichlet_process',
                            'weight_concentration_prior': prior,
                            'max_iter': 1000,
                            'random_state': 42,
                        }
                except Exception as e:
                    print(f'  Skip n={n} cov={cov} prior={prior}: {e}')

    print(f'{dataset_name} BGMM holdout log-likelihood (top 3):')
    for r in sorted(results, key=lambda x: x['val_avg_log_likelihood'], reverse=True)[:3]:
        print(
            f"  n={r['n_components']}, cov={r['covariance_type']}, prior={r['weight_concentration_prior']}: "
            f"score={r['val_avg_log_likelihood']:.4f}"
        )
    print(f'  Best: {best_params}\n')
    return best_params


best_wrought_gmm = tune_gmm_bic(X_wrought, 'Wrought')
best_wrought_bgmm = tune_bgmm_holdout(X_wrought, 'Wrought')

Wrought GMM BIC (top 3):
  n=20, cov=full: BIC=-56134
  n=15, cov=full: BIC=-47276
  n=12, cov=full: BIC=-42195
  Best: {'n_components': 20, 'covariance_type': 'full', 'random_state': 42}



Wrought BGMM holdout log-likelihood (top 3):
  n=12, cov=full, prior=0.05: score=19.8698
  n=12, cov=full, prior=0.1: score=19.8697
  n=12, cov=full, prior=0.2: score=19.8696
  Best: {'n_components': 12, 'covariance_type': 'full', 'weight_concentration_prior_type': 'dirichlet_process', 'weight_concentration_prior': 0.05, 'max_iter': 1000, 'random_state': 42}



In [4]:
# Save backward generator params (wrought only; merge into config)
if best_wrought_gmm is None:
    best_wrought_gmm = get_default_hyperparams('GMM')
    print('No wrought data for GMM tuning; using defaults.')

if best_wrought_bgmm is None:
    best_wrought_bgmm = get_default_hyperparams('BGMM')
    print('No wrought data for BGMM tuning; using defaults.')

save_hyperparams({'backward': {'wrought': {'GMM': best_wrought_gmm, 'BGMM': best_wrought_bgmm}}})
print('Saved backward.wrought.{GMM,BGMM} to hyperparams_config.json')

# Forward per-target params are used by 05_backward_wrought and synthetic generator 06_wrought
by_target_w = load_hyperparams('wrought') and load_hyperparams('wrought').get('by_target')
if by_target_w:
    print('Forward wrought.by_target available for backward/synthetic pipelines.')
else:
    print('Run 01_hyperparameter_tuning_forward.ipynb first for wrought.by_target.')

Saved backward.wrought.{GMM,BGMM} to hyperparams_config.json
Forward wrought.by_target available for backward/synthetic pipelines.
